# Step 1: Explore electricity agreement PDFs

This notebook is our interactive entry point. Run cells in order with **Shift+Enter**.
Select the project **.venv** as the kernel in VS Code. A kernel is the Python process
that remembers variables between cells. Restarting it clears those variables.

We will find PDFs, read one, inspect its pages, and save a structured report.
This stage runs locally and needs no OpenAI API key. The reusable implementation
remains in `electricity_optimizer/`; we call it here so notebook and application agree.

## 1. Check the environment and import our reader
`Path` handles filesystem paths. `read_pdf` is the function we wrote in `ingestion.py`.
The working folder should be the project root, where this notebook lives.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "electricity_optimizer").is_dir():
    raise RuntimeError("Open this notebook from the Electricity_Agreement_Optimizer project root.")

from electricity_optimizer.ingestion import read_pdf
from electricity_optimizer.models import InspectionReport, InspectionFailure

print("Python:", sys.executable)
print("Project:", PROJECT_ROOT)

## 2. Discover the input files
We inspect PDFs directly inside `contracts/`. Sorting makes the order predictable.

In [ ]:
contracts_dir = PROJECT_ROOT / "contracts"
pdf_paths = sorted(p for p in contracts_dir.iterdir() if p.is_file() and p.suffix.lower() == ".pdf")
if not pdf_paths:
    raise FileNotFoundError("Add PDF agreements to contracts/ first.")
for path in pdf_paths:
    print(path.name)

## 3. Read one agreement
Start with the short Reliant document when available. Change `selected_path` to
another file to explore it. PyMuPDF extracts text, and Pydantic organizes it into
`PDFDocument` and `PDFPage` objects. The original PDF is not changed.

In [ ]:
selected_path = next((p for p in pdf_paths if p.name == "R1F00169972621A.pdf"), pdf_paths[0])
document = read_pdf(selected_path)
print("File:", document.source_file)
print("Pages:", len(document.pages))
print("Fingerprint:", document.sha256)

## 4. Inspect a page
Python lists start at index 0, but our source page numbers start at 1.
Find the energy charge, contract length, and cancellation fee in the text.
Compare them with the original PDF: extracted table order can be imperfect.

In [ ]:
page = document.pages[0]
print("Source page:", page.page_number)
print("Needs review:", page.needs_review)
print(page.text)

## 5. Understand structured data
`model_dump()` turns a Pydantic object into a Python dictionary. JSON is the
portable representation we can save to disk. Inspect one page's structure below.
The review flag only detects very sparse text; it does not certify extraction accuracy.

In [ ]:
page.model_dump()

## 6. Inspect every contract
The loop repeats our reader for each PDF. A failed file is recorded while others
continue. This explicit loop shows what the command-line entry point coordinates.

In [ ]:
report = InspectionReport()
for path in pdf_paths:
    try:
        result = read_pdf(path)
    except (OSError, ValueError, RuntimeError) as error:
        report.failures.append(InspectionFailure(source_file=path.name, error=str(error)))
        print("FAILED:", path.name, str(error))
        continue
    report.documents.append(result)
    flagged = [p.page_number for p in result.pages if p.needs_review]
    print(f"{result.source_file}: {len(result.pages)} pages; review pages: {flagged}")

print("Total pages:", sum(len(d.pages) for d in report.documents))
print("Failures:", len(report.failures))

## 7. Save the result
This writes only generated outputs under `output/notebook_inspection/`, keeping
the command-line exports separate. Running again replaces this report.

In [ ]:
output_dir = PROJECT_ROOT / "output" / "notebook_inspection"
output_dir.mkdir(parents=True, exist_ok=True)
report_path = output_dir / "report.json"
report_path.write_text(report.model_dump_json(indent=2), encoding="utf-8")
print("Saved:", report_path)

# Read it back to confirm the saved report matches the in-memory result.
restored = InspectionReport.model_validate_json(report_path.read_text(encoding="utf-8"))
assert restored == report
print("Round-trip validation passed.")

## What you learned and what comes next
- A notebook runs ordinary Python in separate, stateful cells.
- PyMuPDF reads text; Pydantic defines the data structure.
- Page numbers and fingerprints preserve the connection to the source.
- Reading text is separate from interpreting contract terms or calculating costs.

**Next stage:** define agreement fields and extract them with OpenAI, retaining
source citations. Later notebooks will introduce the LangGraph supervisor and
usage-based cost calculations. These documents span different markets and cannot
all be ranked together as competing plans.

After editing imported `.py` files, **restart the kernel and run all cells**.
Clear cell outputs before sharing a notebook containing private agreement data.